In [ ]:
import sys
sys.path.append("../")

import pandas as pd
import numpy as np

from src.archives.heston import estimate_heston_parameters

In [ ]:
portfolio_returns = pd.read_csv(
    "../data/processed/portfolio_returns.csv",
    index_col=0,
    parse_dates=True
).squeeze()

portfolio_returns.name = "Portfolio Return"

portfolio_returns.head()

In [ ]:
heston_params = estimate_heston_parameters(
    portfolio_returns.loc["2015":"2021"]
)

heston_params

In [ ]:
feller_lhs = (
    2
    * heston_params["kappa"]
    * heston_params["theta"]
)

feller_rhs = (
    heston_params["xi"] ** 2
)

print("2κθ =", feller_lhs)
print("ξ²   =", feller_rhs)
print(
    "Feller condition satisfied:",
    feller_lhs > feller_rhs
)

In [ ]:
from src.heston import (
    estimate_heston_parameters,
    simulate_heston
)

In [ ]:
simulated_returns_1d = simulate_heston(
    **heston_params,
    horizon=1,
    n_steps=1,
    n_simulations=50000,
    random_state=42
)

In [ ]:
print(
    "Mean:",
    simulated_returns_1d.mean()
)

print(
    "Std:",
    simulated_returns_1d.std()
)

print(
    "Minimum:",
    simulated_returns_1d.min()
)

print(
    "Maximum:",
    simulated_returns_1d.max()
)

In [ ]:
print(
    "5% quantile:",
    np.quantile(
        simulated_returns_1d,
        0.05
    )
)

print(
    "1% quantile:",
    np.quantile(
        simulated_returns_1d,
        0.01
    )
)

In [ ]:
simulated_returns_10d = simulate_heston(
    **heston_params,
    horizon=10,
    n_steps=10,
    n_simulations=50000,
    random_state=42
)

In [ ]:
print(
    "10-day mean:",
    simulated_returns_10d.mean()
)

print(
    "10-day volatility:",
    simulated_returns_10d.std()
)

print(
    "10-day 5% quantile:",
    np.quantile(
        simulated_returns_10d,
        0.05
    )
)

print(
    "10-day 1% quantile:",
    np.quantile(
        simulated_returns_10d,
        0.01
    )
)

In [ ]:
train_returns = portfolio_returns.loc[
    "2015":"2021"
]

print(
    "Historical annualized volatility:",
    train_returns.std() * np.sqrt(252)
)

print(
    "Heston 1-day simulated volatility:",
    simulated_returns_1d.std() * np.sqrt(252)
)

In [ ]:
pd.DataFrame(
    [heston_params]
).to_csv(
    "../output/tables/heston_training_parameters.csv",
    index=False
)

##### **Heston 1-day Validation**

In [ ]:
from src.archives.heston import (
    walk_forward_heston,
    calculate_var_es
)

In [ ]:
heston_1d_validation = walk_forward_heston(
    portfolio_returns,
    forecast_start="2022-01-03",
    forecast_end="2023-12-29",
    horizon=1,
    n_simulations=50000,
    random_state=42
)

In [ ]:
def future_cumulative_return(
    returns,
    date,
    horizon=10
):
    position = returns.index.get_loc(date)

    future_returns = returns.iloc[
        position + 1:
        position + 1 + horizon
    ]

    if len(future_returns) < horizon:
        return np.nan

    return future_returns.sum()

heston_1d_validation["Realized_Return"] = [
    future_cumulative_return(
        portfolio_returns,
        date,
        horizon=1
    )
    for date in heston_1d_validation.index
]

In [ ]:
heston_1d_validation = (
    heston_1d_validation
    .dropna()
)

In [ ]:
heston_1d_validation["Violation_95"] = (
    heston_1d_validation["Realized_Return"]
    <
    -heston_1d_validation["VaR_95"]
)

heston_1d_validation["Violation_99"] = (
    heston_1d_validation["Realized_Return"]
    <
    -heston_1d_validation["VaR_99"]
)

In [ ]:
print(
    "95% violation rate:",
    heston_1d_validation["Violation_95"].mean()
)

print(
    "99% violation rate:",
    heston_1d_validation["Violation_99"].mean()
)

In [ ]:
from src.risk import kupiec_test, christoffersen_conditional_coverage_test

heston_1d_kupiec_95 = kupiec_test(
    heston_1d_validation["Violation_95"],
    confidence_level=0.95
)

heston_1d_kupiec_99 = kupiec_test(
    heston_1d_validation["Violation_99"],
    confidence_level=0.99
)

heston_1d_cc_95 = (
    christoffersen_conditional_coverage_test(
        heston_1d_validation["Violation_95"],
        confidence_level=0.95
    )
)

heston_1d_cc_99 = (
    christoffersen_conditional_coverage_test(
        heston_1d_validation["Violation_99"],
        confidence_level=0.99
    )
)


heston_1d_backtest = pd.DataFrame([
    {
        "Model": "Heston",
        "Confidence": 0.95,
        **heston_1d_kupiec_95,
        **{
            f"CC_{k}": v
            for k, v in heston_1d_cc_95.items()
        }
    },
    {
        "Model": "Heston",
        "Confidence": 0.99,
        **heston_1d_kupiec_99,
        **{
            f"CC_{k}": v
            for k, v in heston_1d_cc_99.items()
        }
    }
])

heston_1d_backtest

In [ ]:
heston_1d_validation.to_csv(
    "../output/tables/heston_1day_validation_risk.csv"
)

heston_1d_backtest.to_csv(
    "../output/tables/heston_1day_validation_backtest.csv",
    index=False
)

##### **Heston 10-day Validation**

In [ ]:
heston_10d_validation = walk_forward_heston(
    portfolio_returns,
    forecast_start="2022-01-03",
    forecast_end="2023-12-15",
    horizon=10,
    n_simulations=50000,
    random_state=42
)

In [ ]:
heston_10d_validation["Realized_Return_10D"] = [
    future_cumulative_return(
        portfolio_returns,
        date,
        horizon=10
    )
    for date in heston_10d_validation.index
]

heston_10d_validation = (
    heston_10d_validation
    .dropna()
)

In [ ]:
heston_10d_validation["Violation_95"] = (
    heston_10d_validation["Realized_Return_10D"]
    <
    -heston_10d_validation["VaR_95"]
)

heston_10d_validation["Violation_99"] = (
    heston_10d_validation["Realized_Return_10D"]
    <
    -heston_10d_validation["VaR_99"]
)

In [ ]:
print(
    "95% violation rate:",
    heston_10d_validation["Violation_95"].mean()
)

print(
    "99% violation rate:",
    heston_10d_validation["Violation_99"].mean()
)

In [ ]:
heston_10d_kupiec_95 = kupiec_test(
    heston_10d_validation["Violation_95"],
    confidence_level=0.95
)

heston_10d_kupiec_99 = kupiec_test(
    heston_10d_validation["Violation_99"],
    confidence_level=0.99
)

heston_10d_cc_95 = (
    christoffersen_conditional_coverage_test(
        heston_10d_validation["Violation_95"],
        confidence_level=0.95
    )
)

heston_10d_cc_99 = (
    christoffersen_conditional_coverage_test(
        heston_10d_validation["Violation_99"],
        confidence_level=0.99
    )
)


heston_10d_backtest = pd.DataFrame([
    {
        "Model": "Heston",
        "Confidence": 0.95,
        **heston_10d_kupiec_95,
        **{
            f"CC_{k}": v
            for k, v in heston_10d_cc_95.items()
        }
    },
    {
        "Model": "Heston",
        "Confidence": 0.99,
        **heston_10d_kupiec_99,
        **{
            f"CC_{k}": v
            for k, v in heston_10d_cc_99.items()
        }
    }
])

heston_10d_backtest

##### **Performance on Test Set**

In [ ]:
heston_1d_test = walk_forward_heston(
    portfolio_returns,
    forecast_start="2024-01-02",
    forecast_end="2025-12-31",
    horizon=1,
    n_simulations=50000,
    random_state=121
)

In [ ]:
heston_10d_test = walk_forward_heston(
    portfolio_returns,
    forecast_start="2024-01-02",
    forecast_end="2025-12-15",
    horizon=10,
    n_simulations=50000,
    random_state=121
)

In [ ]:
import pandas as pd

# ==========================================
# 1. PROCESS HESTON 1-DAY TEST SET
# ==========================================

# Add realized returns for 1-day horizon
heston_1d_test["Realized_Return"] = [
    future_cumulative_return(portfolio_returns, date, horizon=1)
    for date in heston_1d_test.index
]
heston_1d_test = heston_1d_test.dropna()

# Compute violations
heston_1d_test["Violation_95"] = (
    heston_1d_test["Realized_Return"] < -heston_1d_test["VaR_95"]
)
heston_1d_test["Violation_99"] = (
    heston_1d_test["Realized_Return"] < -heston_1d_test["VaR_99"]
)

# Run backtests
heston_1d_kupiec_95 = kupiec_test(
    heston_1d_test["Violation_95"], confidence_level=0.95
)
heston_1d_kupiec_99 = kupiec_test(
    heston_1d_test["Violation_99"], confidence_level=0.99
)

heston_1d_cc_95 = christoffersen_conditional_coverage_test(
    heston_1d_test["Violation_95"], confidence_level=0.95
)
heston_1d_cc_99 = christoffersen_conditional_coverage_test(
    heston_1d_test["Violation_99"], confidence_level=0.99
)

# Build summary table for 1-day test backtest
heston_1d_test_backtest = pd.DataFrame([
    {
        "Model": "Heston (1-Day)",
        "Confidence": 0.95,
        **heston_1d_kupiec_95,
        **{f"CC_{k}": v for k, v in heston_1d_cc_95.items()},
    },
    {
        "Model": "Heston (1-Day)",
        "Confidence": 0.99,
        **heston_1d_kupiec_99,
        **{f"CC_{k}": v for k, v in heston_1d_cc_99.items()},
    },
])


# ==========================================
# 2. PROCESS HESTON 10-DAY TEST SET
# ==========================================

# Add realized returns for 10-day horizon
heston_10d_test["Realized_Return_10D"] = [
    future_cumulative_return(portfolio_returns, date, horizon=10)
    for date in heston_10d_test.index
]
heston_10d_test = heston_10d_test.dropna()

# Compute violations
heston_10d_test["Violation_95"] = (
    heston_10d_test["Realized_Return_10D"] < -heston_10d_test["VaR_95"]
)
heston_10d_test["Violation_99"] = (
    heston_10d_test["Realized_Return_10D"] < -heston_10d_test["VaR_99"]
)

# Run backtests
heston_10d_kupiec_95 = kupiec_test(
    heston_10d_test["Violation_95"], confidence_level=0.95
)
heston_10d_kupiec_99 = kupiec_test(
    heston_10d_test["Violation_99"], confidence_level=0.99
)

heston_10d_cc_95 = christoffersen_conditional_coverage_test(
    heston_10d_test["Violation_95"], confidence_level=0.95
)
heston_10d_cc_99 = christoffersen_conditional_coverage_test(
    heston_10d_test["Violation_99"], confidence_level=0.99
)

# Build summary table for 10-day test backtest
heston_10d_test_backtest = pd.DataFrame([
    {
        "Model": "Heston (10-Day)",
        "Confidence": 0.95,
        **heston_10d_kupiec_95,
        **{f"CC_{k}": v for k, v in heston_10d_cc_95.items()},
    },
    {
        "Model": "Heston (10-Day)",
        "Confidence": 0.99,
        **heston_10d_kupiec_99,
        **{f"CC_{k}": v for k, v in heston_10d_cc_99.items()},
    },
])


# ==========================================
# 3. SAVE RESULTS TO OUTPUT TABLES
# ==========================================

heston_1d_test.to_csv("../output/tables/heston_1day_test_risk.csv")
heston_1d_test_backtest.to_csv(
    "../output/tables/heston_1day_test_backtest.csv", index=False
)

heston_10d_test.to_csv("../output/tables/heston_10day_test_risk.csv")
heston_10d_test_backtest.to_csv(
    "../output/tables/heston_10day_test_backtest.csv", index=False
)

# Display backtest summaries
display(heston_1d_test_backtest)
display(heston_10d_test_backtest)

In [ ]:
from src.risk import create_backtest_summary

In [ ]:
heston_1d_summary = create_backtest_summary(
    heston_1d_test,
    model="Heston",
    horizon=1
)